# Personal Career Agent
**Interview Preparation Assistant — powered by Groq + Gradio**

- Generate tailored interview questions from resume + job description
- Practice answering in a chat interface
- Receive structured feedback and scores (1–10) per answer
- Session summary with overall performance


In [ ]:
# Cell 2: Imports
import os
import sys
import json
import asyncio
import gradio as gr
from groq import Groq
from dotenv import load_dotenv


print(f"Gradio version : {gr.__version__}")
print(f"Python version : {sys.version}")
print(f"Platform       : {sys.platform}")

In [ ]:
# Cell 3: Constants
# Windows + Python 3.12 fix: prevents asyncio event-loop mismatch in Gradio 5 / uvicorn
load_dotenv(override=True)
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    print("WindowsSelectorEventLoopPolicy set.")

GROQ_API_KEY  = os.getenv("GROQ_API_KEY")
MODEL         = "llama-3.3-70b-versatile"
NUM_QUESTIONS = 2
MAX_TOKENS    = 1024

print(f"API key present: {GROQ_API_KEY != 'your-groq-api-key-here'}")

In [ ]:
# Cell 2.5: Resume file parser

import fitz                 # pymupdf — PDF parsing
from docx import Document   # python-docx — DOCX parsing


def extract_path(file_input) -> str | None:
    """Safely extract a file path from any Gradio 5 file input variant."""
    if file_input is None:
        return None
    if isinstance(file_input, str):      # plain str or NamedString (str subclass)
        return file_input
    if isinstance(file_input, dict):     # legacy dict form
        return file_input.get("name")
    if hasattr(file_input, "name"):      # object with .name attribute
        return file_input.name
    return str(file_input)


def parse_resume(file_input) -> str:
    """Extract plain text from an uploaded PDF or DOCX resume."""
    path = extract_path(file_input)
    if not path:
        raise ValueError("No file provided.")

    ext = os.path.splitext(path)[-1].lower()

    if ext == ".pdf":
        doc  = fitz.open(path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        return text.strip()

    elif ext == ".docx":
        doc  = Document(path)
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
        return text.strip()

    else:
        raise ValueError(f"Unsupported file type '{ext}'. Please upload a PDF or DOCX.")


print("Resume parser defined.")

In [ ]:
# Cell 4: Tool schemas
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "generate_questions",
            "description": "Generate tailored interview questions based on the candidate's resume and job description.",
            "parameters": {
                "type": "object",
                "properties": {
                    "questions": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": f"List of exactly {NUM_QUESTIONS} interview questions."
                    }
                },
                "required": ["questions"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "evaluate_answer",
            "description": "Evaluate a candidate's answer and return a score with structured feedback.",
            "parameters": {
                "type": "object",
                "properties": {
                    "score": {
                        "type": "integer",
                        "description": "Score from 1 (very poor) to 10 (excellent)."
                    },
                    "strengths": {
                        "type": "string",
                        "description": "What the candidate did well."
                    },
                    "improvements": {
                        "type": "string",
                        "description": "Specific, actionable suggestions to improve the answer."
                    },
                    "ideal_answer_hint": {
                        "type": "string",
                        "description": "A brief pointer toward what an ideal answer would cover."
                    }
                },
                "required": ["score", "strengths", "improvements", "ideal_answer_hint"]
            }
        }
    }
]

print(f"Tools defined: {[t['function']['name'] for t in TOOLS]}")

In [ ]:
# Cell 5: Tool execution functions

def execute_generate_questions(args: dict, session: dict) -> str:
    questions = args.get("questions", [])
    if not questions:
        return "Could not generate questions. Please check your resume and job description."

    session["questions"]       = questions
    session["current_index"]   = 0
    session["answers"]         = []
    session["evaluations"]     = []
    session["awaiting_answer"] = True

    lines = [
        f"Great! I've prepared **{len(questions)} interview questions** tailored to your profile.\n",
        "Let's begin. Take your time with each answer.\n",
        f"**Question 1 of {len(questions)}:**\n",
        questions[0]
    ]
    return "\n".join(lines)


def execute_evaluate_answer(args: dict, session: dict) -> str:
    score        = args.get("score", 0)
    strengths    = args.get("strengths", "")
    improvements = args.get("improvements", "")
    ideal_hint   = args.get("ideal_answer_hint", "")

    session["evaluations"].append({
        "question":     session["questions"][session["current_index"]],
        "answer":       session["answers"][-1],
        "score":        score,
        "strengths":    strengths,
        "improvements": improvements,
        "ideal_hint":   ideal_hint
    })

    feedback_lines = [
        f"**Score: {score}/10**\n",
        f"**Strengths:** {strengths}\n",
        f"**Improve:** {improvements}\n",
        f"**Ideal answer should cover:** {ideal_hint}\n"
    ]

    session["current_index"] += 1
    next_idx = session["current_index"]
    total    = len(session["questions"])

    if next_idx < total:
        feedback_lines.append(f"---\n**Question {next_idx + 1} of {total}:**\n")
        feedback_lines.append(session["questions"][next_idx])
    else:
        session["awaiting_answer"] = False
        scores = [e["score"] for e in session["evaluations"]]
        avg    = sum(scores) / len(scores)
        feedback_lines.append("---\n## Session Complete!\n")
        feedback_lines.append(f"**Overall Average Score: {avg:.1f}/10**\n")
        for i, ev in enumerate(session["evaluations"]):
            feedback_lines.append(f"**Q{i+1}:** {ev['question']}")
            feedback_lines.append(f"  - Score: {ev['score']}/10 | Improve: {ev['improvements']}\n")
        feedback_lines.append("\nType **restart** to begin a new session.")

    return "\n".join(feedback_lines)


print("Tool functions defined.")

In [ ]:
# Cell 6: Tool dispatcher

def dispatch_tool(tool_name: str, tool_args: dict, session: dict) -> str:
    print(f"  [DEBUG] dispatch_tool: {tool_name}")
    if tool_name == "generate_questions":
        return execute_generate_questions(tool_args, session)
    elif tool_name == "evaluate_answer":
        return execute_evaluate_answer(tool_args, session)
    return f"Unknown tool: {tool_name}"


print("Dispatcher defined.")

In [ ]:
# Cell 7: LLM call
# Client instantiated per-call — avoids asyncio event-loop conflicts on Windows

def call_llm(messages: list, use_tools: bool = True) -> object:
    client = Groq(api_key=GROQ_API_KEY)
    kwargs = {
        "model":      MODEL,
        "messages":   messages,
        "max_tokens": MAX_TOKENS,
    }
    if use_tools:
        kwargs["tools"]       = TOOLS
        kwargs["tool_choice"] = "auto"

    print(f"  [DEBUG] call_llm: {len(messages)} messages, tools={use_tools}")
    response = client.chat.completions.create(**kwargs)
    msg = response.choices[0].message
    print(f"  [DEBUG] call_llm response: tool_calls={bool(msg.tool_calls)}, content_len={len(msg.content or '')}")
    return msg


print("LLM call defined.")

In [ ]:
# Cell 8: System prompt builder

def build_system_prompt(session: dict) -> str:
    return f"""You are a professional interview coach and career advisor.

Your job is to help candidates prepare for job interviews through realistic practice.

You have access to two tools:
1. generate_questions — call this ONCE at the start to create {NUM_QUESTIONS} targeted interview questions.
2. evaluate_answer   — call this each time the candidate submits an answer.

CANDIDATE RESUME:
{session.get('resume', '')}

JOB DESCRIPTION:
{session.get('job_description', '')}

RULES:
- Generate questions specific to the role and the candidate's background.
- Mix behavioral (STAR-method), technical, and situational questions.
- Evaluations must be honest, fair, and constructive.
- Keep feedback concise and actionable.
- Ask one question at a time — never reveal all questions upfront.
- Never break character or discuss anything outside interview preparation.
"""


print("System prompt builder defined.")

In [ ]:
# Cell 9: Agent orchestrator loop

def run_agent(user_message: str, session: dict) -> str:
    """
    Agentic loop (no framework):
      1. Append user message to history.
      2. Call LLM.
      3. Tool call returned -> dispatch -> feed result back -> return.
      4. No tool call -> return plain text reply.
    """
    print(f"  [DEBUG] run_agent: '{user_message[:60]}'")

    history  = session.setdefault("history", [])
    history.append({"role": "user", "content": user_message})
    messages = [{"role": "system", "content": build_system_prompt(session)}] + history

    while True:
        response_msg = call_llm(messages, use_tools=True)

        if not response_msg.tool_calls:
            reply = response_msg.content or ""
            history.append({"role": "assistant", "content": reply})
            print(f"  [DEBUG] run_agent: plain reply len={len(reply)}")
            return reply

        tool_call   = response_msg.tool_calls[0]
        tool_name   = tool_call.function.name
        tool_args   = json.loads(tool_call.function.arguments)
        tool_result = dispatch_tool(tool_name, tool_args, session)

        messages.append(response_msg)
        messages.append({
            "role":         "tool",
            "tool_call_id": tool_call.id,
            "content":      tool_result
        })
        history.append({"role": "assistant", "content": tool_result})
        print(f"  [DEBUG] run_agent: tool result len={len(tool_result)}")
        return tool_result


print("Agent loop defined.")

In [ ]:
# Cell 10: Gradio UI

def create_session():
    return {
        "resume":          "",
        "job_description": "",
        "questions":       [],
        "current_index":   0,
        "answers":         [],
        "evaluations":     [],
        "awaiting_answer": False,
        "history":         []
    }


def start_session(resume_file, job_description: str, session: dict):
    print("[DEBUG] start_session called")
    print(f"[DEBUG] resume_file : {resume_file!r}")
    print(f"[DEBUG] jd length   : {len(job_description)}")

    if resume_file is None:
        return session, [(None, "Please upload your resume (PDF or DOCX) before starting.")]
    if not job_description.strip():
        return session, [(None, "Please paste the job description before starting.")]

    try:
        resume_text = parse_resume(resume_file)
        print(f"[DEBUG] resume parsed, length={len(resume_text)}")
    except ValueError as e:
        print(f"[DEBUG] parse error: {e}")
        return session, [(None, str(e))]
    except Exception as e:
        print(f"[DEBUG] parse unexpected error: {type(e).__name__}: {e}")
        return session, [(None, f"Could not read resume: {type(e).__name__}: {e}")]

    if not resume_text:
        return session, [(None, "Resume appears to be empty. Please check the file and try again.")]

    try:
        session.update(create_session())
        session["resume"]          = resume_text
        session["job_description"] = job_description.strip()
        print("[DEBUG] start_session: calling run_agent...")
        response = run_agent("Please generate my interview questions and start the session.", session)
        print(f"[DEBUG] start_session: response len={len(response)}")
        return session, [(None, response)]
    except Exception as e:
        print(f"[DEBUG] start_session ERROR: {type(e).__name__}: {e}")
        return session, [(None, f"Error: {type(e).__name__}: {e}")]


def chat(user_input: str, chat_history: list, session: dict):
    print(f"[DEBUG] chat called: '{user_input[:60]}'")

    if not user_input.strip():
        return "", chat_history, session

    if user_input.strip().lower() == "restart":
        session.update(create_session())
        chat_history.append((user_input, "Session reset. Upload a new resume and click **Start Interview**."))
        return "", chat_history, session

    if not session.get("questions"):
        chat_history.append((user_input, "Please click **Start Interview** first to generate your questions."))
        return "", chat_history, session

    try:
        if session.get("awaiting_answer"):
            session["answers"].append(user_input.strip())
            eval_prompt = (
                f"The candidate just answered question {session['current_index'] + 1}: "
                f"'{session['questions'][session['current_index']]}'. "
                f"Their answer: '{user_input.strip()}'. "
                f"Please evaluate this answer using the evaluate_answer tool."
            )
            response = run_agent(eval_prompt, session)
        else:
            response = run_agent(user_input.strip(), session)

        chat_history.append((user_input, response))
    except Exception as e:
        print(f"[DEBUG] chat ERROR: {type(e).__name__}: {e}")
        chat_history.append((user_input, f"Error: {type(e).__name__}: {e}"))

    return "", chat_history, session


# ── Layout ───────────────────────────────────────────────────────────
with gr.Blocks(title="Personal Career Agent", theme=gr.themes.Soft()) as app:

    gr.Markdown("# Personal Career Agent\n*AI-powered interview preparation*")

    session_state = gr.State(create_session)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Your Profile")
            resume_file = gr.File(
                label="Resume (PDF or DOCX)",
                file_types=[".pdf", ".docx"],
                type="filepath"
            )
            jd_input = gr.Textbox(
                label="Job Description",
                placeholder="Paste the job description here...",
                lines=12
            )
            start_btn = gr.Button("Start Interview", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### Interview Session")
            chatbot = gr.Chatbot(height=480, show_label=False)
            with gr.Row():
                user_input = gr.Textbox(
                    label="Your Answer",
                    placeholder="Type your answer and press Enter...",
                    lines=3,
                    scale=4
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)

    start_btn.click(
        fn=start_session,
        inputs=[resume_file, jd_input, session_state],
        outputs=[session_state, chatbot]
    )
    send_btn.click(
        fn=chat,
        inputs=[user_input, chatbot, session_state],
        outputs=[user_input, chatbot, session_state]
    )
    user_input.submit(
        fn=chat,
        inputs=[user_input, chatbot, session_state],
        outputs=[user_input, chatbot, session_state]
    )

print("UI built successfully.")

In [ ]:
# Cell 11: Launch

app.launch(share=False, inbrowser=True)